In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)      # fixed seed => repeatable
WINDOWS = 60                        # each run = 60 windows of 10 seconds

# Typical behaviour per 10-second window for each kind of benign activity.
# mod=files modified, ren=files renamed, extp=share of renames that CHANGE the extension,
# ent=(mean, std) of write entropy (0-8; encrypted/compressed data is near 8),
# dirs=folders touched, dele=files deleted, mb=megabytes written
BENIGN = {
    "office":   dict(mod=8,   ren=1,  extp=0.10, ent=(4.5, 1.0), dirs=3,  dele=1,  mb=3),
    "backup":   dict(mod=120, ren=2,  extp=0.05, ent=(6.5, 0.8), dirs=20, dele=2,  mb=120),
    "compress": dict(mod=60,  ren=30, extp=0.60, ent=(7.7, 0.2), dirs=6,  dele=5,  mb=60),   # confounder!
    "update":   dict(mod=80,  ren=40, extp=0.30, ent=(6.0, 0.8), dirs=30, dele=10, mb=50),   # confounder!
}

def ransom_params(stealth):
    # stealth=1.0 is loud ransomware; lower values mean throttled (slower, harder to see)
    return dict(mod=150*stealth + 10, ren=120*stealth, extp=0.95, ent=(7.6, 0.2),
                dirs=25*stealth + 3, dele=20*stealth, mb=80*stealth + 5)

def one_window(p):
    mod, ren = rng.poisson(p["mod"]), rng.poisson(p["ren"])
    return dict(
        files_modified=mod,
        files_renamed=ren,
        ext_changed=rng.binomial(ren, p["extp"]),                  # renames that changed the extension
        mean_entropy=float(np.clip(rng.normal(*p["ent"]), 0, 8)),
        dirs_touched=rng.poisson(p["dirs"]),
        files_deleted=rng.poisson(p["dele"]),
        mb_written=rng.poisson(p["mb"]),
    )

def make_run(run_id, kind, stealth=1.0):
    start = int(rng.integers(10, 40)) if kind == "ransomware" else -1   # when infection begins
    rows = []
    for w in range(WINDOWS):
        infected = kind == "ransomware" and w >= start
        if infected:
            p = ransom_params(stealth)            # ransomware behaviour
        elif kind == "ransomware":
            p = BENIGN["office"]                  # before infection it is a normal office host
        else:
            p = BENIGN[kind]
        r = one_window(p)
        r.update(run_id=run_id, kind=kind, window=w, start=start, label=int(infected))
        rows.append(r)
    return rows

def make_dataset(n_per_benign=75, n_ransom=60, stealth=1.0, offset=0):
    rows, rid = [], offset
    for kind in BENIGN:
        for _ in range(n_per_benign):
            rows += make_run(rid, kind); rid += 1
    for _ in range(n_ransom):
        rows += make_run(rid, "ransomware", stealth); rid += 1
    return pd.DataFrame(rows)

df = make_dataset()
print(df.shape, "| malicious windows:", df["label"].sum(), f"({df['label'].mean():.1%})")
print(df.groupby("kind")[["files_modified", "ext_changed", "mean_entropy"]].mean().round(1))

(21600, 12) | malicious windows: 2190 (10.1%)
            files_modified  ext_changed  mean_entropy
kind                                                 
backup               120.3          0.1           6.5
compress              60.3         18.0           7.7
office                 8.0          0.1           4.5
ransomware           100.3         69.2           6.4
update                79.8         12.0           6.0


In [2]:
df = df.sort_values(["run_id", "window"]).reset_index(drop=True)
hist_cols = ["files_modified", "ext_changed", "mean_entropy"]
prev = (df.groupby("run_id")[hist_cols]
          .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
          .add_suffix("_prev3").fillna(0))
df = pd.concat([df, prev], axis=1)
feats = ["files_modified", "files_renamed", "ext_changed", "mean_entropy",
         "dirs_touched", "files_deleted", "mb_written"] + list(prev.columns)

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix
from xgboost import XGBClassifier

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(df, df["label"], groups=df["run_id"]))
train, test = df.iloc[tr_idx], df.iloc[te_idx].copy()
print("ransomware runs in test:", test.loc[test["kind"] == "ransomware", "run_id"].nunique())

spw = (train["label"] == 0).sum() / (train["label"] == 1).sum()   # class-imbalance weight
model = XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=spw,
                      n_jobs=-1, random_state=42).fit(train[feats], train["label"])

def report(m, d, name, thr=0.5):
    p = m.predict_proba(d[feats])[:, 1]
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(d["label"], pred).ravel()
    print(f"{name}: PR-AUC={average_precision_score(d['label'], p):.4f} "
          f"recall={tp/(tp+fn):.4f} precision={tp/max(tp+fp,1):.4f} FPR={fp/(fp+tn):.5f} (FP={fp}, FN={fn})")
    return p

test["score"] = report(model, test, "XGBoost, loud ransomware")

ransomware runs in test: 20
XGBoost, loud ransomware: PR-AUC=1.0000 recall=1.0000 precision=1.0000 FPR=0.00000 (FP=0, FN=0)


In [6]:
test["alert"] = (test["score"] >= 0.5).astype(int)

# (a) detection delay: windows between infection start and the first alert
lat = []
for rid, g in test[test["kind"] == "ransomware"].groupby("run_id"):
    first = g.loc[(g["alert"] == 1) & (g["label"] == 1), "window"]
    lat.append(first.min() - g["start"].iloc[0] if len(first) else np.nan)
print("runs never detected:", int(np.isnan(lat).sum()), "| median delay (windows):", np.nanmedian(lat))

# (b) false alarm rate per benign kind
print(test[test["label"] == 0].groupby("kind")["alert"].mean().round(4))

runs never detected: 0 | median delay (windows): 0.0
kind
backup        0.0
compress      0.0
office        0.0
ransomware    0.0
update        0.0
Name: alert, dtype: float64


In [4]:
for s in [0.5, 0.2]:
    stealthy = make_dataset(stealth=s, offset=100000)       # new runs, new ids
    stealthy = stealthy.sort_values(["run_id", "window"]).reset_index(drop=True)
    pv = (stealthy.groupby("run_id")[hist_cols]
                  .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
                  .add_suffix("_prev3").fillna(0))
    stealthy = pd.concat([stealthy, pv], axis=1)
    report(model, stealthy, f"stealth={s}")

stealth=0.5: PR-AUC=0.0996 recall=0.0005 precision=1.0000 FPR=0.00000 (FP=0, FN=2142)
stealth=0.2: PR-AUC=0.1011 recall=0.0000 precision=0.0000 FPR=0.00000 (FP=0, FN=2183)


In [9]:
import os, joblib
os.makedirs("../data/processed", exist_ok=True); os.makedirs("../models", exist_ok=True)
test[["run_id", "window", "label", "score"]].to_csv("../data/processed/ransomware_scores.csv", index=False)
joblib.dump(model, "../models/ransomware_xgb.joblib"); joblib.dump(feats, "../models/ransomware_features.joblib")

['../models/ransomware_features.joblib']

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

# ---------- 1) a closer confounder: bulk renaming (e.g. photo organiser) ----------
BENIGN["bulk_rename"] = dict(mod=40, ren=60, extp=0.8, ent=(5.5, 0.8), dirs=8, dele=3, mb=20)

# ---------- 2) fairer runs: EVERY run starts as plain office work ----------
def make_run2(run_id, kind, stealth=1.0):
    start = int(rng.integers(12, 40))              # when the special activity begins (after >=12 office windows)
    rows = []
    for w in range(WINDOWS):
        active = (w >= start) and (kind != "office")
        if active:
            p = ransom_params(stealth) if kind == "ransomware" else BENIGN[kind]
        else:
            p = BENIGN["office"]                    # before 'start', every host looks like office work
        r = one_window(p)
        # label=1 ONLY for ransomware windows. Compress/backup/update/bulk_rename stay label 0.
        r.update(run_id=run_id, kind=kind, window=w, start=start,
                 label=int(active and kind == "ransomware"))
        rows.append(r)
    return rows

def make_dataset2(n_per_benign=40, n_ransom=40, stealth=1.0, offset=0):
    rows, rid = [], offset
    for kind in BENIGN:                             # office, backup, compress, update, bulk_rename
        for _ in range(n_per_benign):
            rows += make_run2(rid, kind); rid += 1
    for _ in range(n_ransom):
        rows += make_run2(rid, "ransomware", stealth); rid += 1
    return pd.DataFrame(rows)

# ---------- 3) feature builder ----------
def add_features(d):
    d = d.sort_values(["run_id", "window"]).reset_index(drop=True)
    base_cols = ["files_modified", "files_renamed", "ext_changed", "mean_entropy",
                 "dirs_touched", "files_deleted", "mb_written"]
    g = d.groupby("run_id")
    # host baseline = average of the first 10 windows (all runs are plain office work then)
    base = g[base_cols].transform(lambda s: s.iloc[:10].mean())
    dev = (d[base_cols] - base).add_suffix("_dev")          # how far this window is from the host's own normal
    # sustained activity = sum over the last 6 windows (including now); slow ransomware adds up
    sus = (g[["ext_changed", "files_modified", "files_renamed"]]
             .transform(lambda s: s.rolling(6, min_periods=1).sum()).add_suffix("_sum6"))
    out = pd.concat([d, dev, sus], axis=1)
    return out, base_cols, base_cols + list(dev.columns) + list(sus.columns)

# ---------- 4) mix loud + stealthy ransomware in one dataset ----------
parts = []
for i, s in enumerate([1.0, 0.5, 0.2]):
    d = make_dataset2(stealth=s, offset=i * 10000)
    d["stealth"] = np.where(d["kind"] == "ransomware", s, 0.0)   # 0.0 marks non-ransomware runs
    parts.append(d)
df2 = pd.concat(parts, ignore_index=True)
df2, single_cols, all_cols = add_features(df2)
print(df2.shape, "| malicious windows:", df2["label"].sum(), f"({df2['label'].mean():.1%})")

(43200, 23) | malicious windows: 4072 (9.4%)


In [8]:
# split by RUN so neighbouring windows of one run never land in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(df2, df2["label"], groups=df2["run_id"]))
train, test = df2.iloc[tr_idx], df2.iloc[te_idx].copy()
spw = (train["label"] == 0).sum() / (train["label"] == 1).sum()

for name, cols in {"single-window": single_cols, "+baseline & sustained": all_cols}.items():
    m = XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=spw,
                      n_jobs=-1, random_state=42).fit(train[cols], train["label"])
    test["score"] = m.predict_proba(test[cols])[:, 1]

    # pick the alert threshold so that only ~1% of benign windows raise an alert
    thr = np.quantile(test.loc[test["label"] == 0, "score"], 0.99)
    test["hit"] = (test["score"] >= thr).astype(int)

    print(f"\n=== {name} === PR-AUC={average_precision_score(test['label'], test['score']):.3f}")
    print("recall at 1% FPR, by ransomware stealth level:")
    print(test[test["label"] == 1].groupby("stealth")["hit"].mean().round(3))
    print("false-alarm rate by benign type:")
    print(test[test["label"] == 0].groupby("kind")["hit"].mean().round(4))


=== single-window === PR-AUC=1.000
recall at 1% FPR, by ransomware stealth level:
stealth
0.2    1.0
0.5    1.0
1.0    1.0
Name: hit, dtype: float64
false-alarm rate by benign type:
kind
backup         0.0012
bulk_rename    0.0181
compress       0.0256
office         0.0013
ransomware     0.0000
update         0.0027
Name: hit, dtype: float64

=== +baseline & sustained === PR-AUC=1.000
recall at 1% FPR, by ransomware stealth level:
stealth
0.2    1.0
0.5    1.0
1.0    1.0
Name: hit, dtype: float64
false-alarm rate by benign type:
kind
backup         0.0012
bulk_rename    0.0105
compress       0.0315
office         0.0006
ransomware     0.0011
update         0.0027
Name: hit, dtype: float64


In [10]:
# reuse the run-based split (train/test) from Step 7
train_h = train[train["stealth"] != 0.2]      # remove stealth-0.2 ransomware runs from TRAINING only
spw_h = (train_h["label"] == 0).sum() / (train_h["label"] == 1).sum()
m = XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=spw_h,
                  n_jobs=-1, random_state=42).fit(train_h[all_cols], train_h["label"])
test["score"] = m.predict_proba(test[all_cols])[:, 1]

thr = np.quantile(test.loc[test["label"] == 0, "score"], 0.99)   # 1% false-alarm budget
test["hit"] = (test["score"] >= thr).astype(int)
print("recall at 1% FPR by stealth (0.2 was never seen in training):")
print(test[test["label"] == 1].groupby("stealth")["hit"].mean().round(3))

recall at 1% FPR by stealth (0.2 was never seen in training):
stealth
0.2    0.059
0.5    1.000
1.0    1.000
Name: hit, dtype: float64


In [11]:
import os

# retrain the NORMAL model: all stealth levels in training (not the held-out one)
final_model = XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=spw,
                            n_jobs=-1, random_state=42).fit(train[all_cols], train["label"])

# overwrite test["score"] with this model's scores
test["score"] = final_model.predict_proba(test[all_cols])[:, 1]

# quick check: how many windows would raise an alert at 0.5, and how many are real?
alerts = test[test["score"] >= 0.5]
print("alerts:", len(alerts), "| real ransomware among them:", int(alerts["label"].sum()))

# save for Phase 4
os.makedirs("../data/processed", exist_ok=True)
test[["run_id", "window", "kind", "label", "score"]].to_csv(
    "../data/processed/ransomware_scores.csv", index=False)

alerts: 1275 | real ransomware among them: 1274


In [12]:
import glob
print(glob.glob("../data/processed/*.csv"))

['../data/processed\\insider_scores.csv', '../data/processed\\network_scores.csv', '../data/processed\\ransomware_scores.csv']
